In [1]:
import numpy as np
import pandas as pd
import catboost as cat
from omegaconf import OmegaConf
from sklearn.model_selection import train_test_split

In [181]:
raw_features_df = pd.read_csv('../../work/input/drivendata/flu-shot/training_set_features.csv', engine='pyarrow')
raw_labels_df = pd.read_csv('../../work/input/drivendata/flu-shot/training_set_labels.csv', engine='pyarrow')
test_features_df = pd.read_csv('../../work/input/drivendata/flu-shot/test_set_features.csv', engine='pyarrow')
raw_train_df = pd.merge(features_df, labels_df, how='inner', on='respondent_id')

In [166]:

labels_vaccines = np.where( (raw_train_df.h1n1_vaccine==0) & raw_train_df.seasonal_vaccine==0,0,
                np.where( (raw_train_df.h1n1_vaccine==0) & raw_train_df.seasonal_vaccine==1, 1,
                        np.where((raw_train_df.h1n1_vaccine==1) & raw_train_df.seasonal_vaccine==0,2,
                                np.where((raw_train_df.h1n1_vaccine==1) & raw_train_df.seasonal_vaccine==1,3,np.nan)
                )
        )
).astype('int')

In [189]:
from sklearn.model_selection import train_test_split

train_df, valid_df = train_test_split(raw_train_df.assign(labels_vaccines=labels_vaccines), train_size=0.8, shuffle=True)

In [169]:
cf = features_df.select_dtypes(include='object').columns.to_list()
cf

['age_group',
 'education',
 'race',
 'sex',
 'income_poverty',
 'marital_status',
 'rent_or_own',
 'employment_status',
 'hhs_geo_region',
 'census_msa',
 'employment_industry',
 'employment_occupation']

In [179]:
bast_params = {
    'objective': 'MultiClass',
    #'eval_metric': 'MultiClass',
    'nan_mode': 'Min',
    'l2_leaf_reg': 2.8,
    "eval_metric": "AUC",
    "random_seed": 12,
    "task_type": "CPU",
}

In [192]:
train_df.replace({None: np.nan}, inplace=True)
train_df[cf]=features_df[cf].replace({np.nan: 'null'})
train_df.isnull().sum().sort_values(ascending=False)

health_insurance               9794
doctor_recc_h1n1               1721
doctor_recc_seasonal           1721
chronic_med_condition           789
child_under_6_months            672
health_worker                   658
opinion_seas_sick_from_vacc     436
opinion_seas_risk               419
opinion_seas_vacc_effective     373
opinion_h1n1_sick_from_vacc     329
opinion_h1n1_risk               323
opinion_h1n1_vacc_effective     318
household_adults                199
household_children              199
behavioral_avoidance            169
behavioral_touch_face           100
h1n1_knowledge                   98
h1n1_concern                     70
behavioral_large_gatherings      65
behavioral_outside_home          63
behavioral_antiviral_meds        54
behavioral_wash_hands            36
behavioral_face_mask             16
hhs_geo_region                    0
seasonal_vaccine                  0
h1n1_vaccine                      0
employment_occupation             0
employment_industry         

In [196]:
model = cat.CatBoostClassifier(**bast_params, iterations=1000)
model.fit(
    X=train_df.drop(['h1n1_vaccine','seasonal_vaccine','labels_vaccines','respondent_id'], axis=1), 
    #y=train_df[['h1n1_vaccine','seasonal_vaccine']], 
    y=train_df['labels_vaccines'],
    cat_features=cf, 
    eval_set=[(valid_df.drop(['h1n1_vaccine','seasonal_vaccine','labels_vaccines','respondent_id'], axis=1), valid_df.labels_vaccines)], 
    verbose=100, use_best_model=True, plot=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	test: 0.7662886	best: 0.7662886 (0)	total: 82.5ms	remaining: 1m 22s
100:	test: 0.8106910	best: 0.8106910 (100)	total: 2.17s	remaining: 19.3s
200:	test: 0.8163129	best: 0.8163421 (199)	total: 3.79s	remaining: 15.1s
300:	test: 0.8181556	best: 0.8181672 (286)	total: 5.17s	remaining: 12s
400:	test: 0.8200474	best: 0.8200474 (400)	total: 6.56s	remaining: 9.8s
500:	test: 0.8213154	best: 0.8213154 (500)	total: 7.96s	remaining: 7.93s
600:	test: 0.8218429	best: 0.8218599 (597)	total: 9.41s	remaining: 6.25s
700:	test: 0.8222289	best: 0.8222525 (693)	total: 10.9s	remaining: 4.64s
800:	test: 0.8224449	best: 0.8224449 (800)	total: 12.3s	remaining: 3.06s
900:	test: 0.8228215	best: 0.8228215 (900)	total: 13.8s	remaining: 1.51s
999:	test: 0.8230449	best: 0.8230657 (989)	total: 15.3s	remaining: 0us

bestTest = 0.823065726
bestIteration = 989

Shrink model to first 990 iterations.


In [205]:
test_features_df.replace({None: np.nan}, inplace=True)
test_features_df[cf]=test_features_df[cf].replace({np.nan: 'null'})
test_features_df.isnull().sum().sort_values(ascending=False)

health_insurance               12228
doctor_recc_seasonal            2160
doctor_recc_h1n1                2160
chronic_med_condition            932
child_under_6_months             813
health_worker                    789
opinion_seas_sick_from_vacc      521
opinion_seas_risk                499
opinion_seas_vacc_effective      452
opinion_h1n1_vacc_effective      398
opinion_h1n1_risk                380
opinion_h1n1_sick_from_vacc      375
household_children               225
household_adults                 225
behavioral_avoidance             213
behavioral_touch_face            128
h1n1_knowledge                   122
h1n1_concern                      85
behavioral_outside_home           82
behavioral_antiviral_meds         79
behavioral_large_gatherings       72
behavioral_wash_hands             40
behavioral_face_mask              19
rent_or_own                        0
employment_industry                0
census_msa                         0
hhs_geo_region                     0
e

In [209]:
test_features_df.respondent_id.values

array([26707, 26708, 26709, ..., 53412, 53413, 53414])

In [210]:
test_features_cat = cat.Pool(data=test_features_df, cat_features=cf)
result = model.predict(test_features_cat)

In [215]:
pd.DataFrame(np.column_stack((test_features_df.respondent_id.values, result)))

,0,1
0,26707,0
1,26708,0
2,26709,0
3,26710,0
4,26711,0
...,...,...
26703,53410,0
26704,53411,0
26705,53412,0
26706,53413,0
